# 08 Derivations (LO3)

Three derivations, each followed by a numerical check on our own data so the algebra is tied to the models we actually use:

1. **Logistic regression** (our chosen classifier): log-loss, gradient and Hessian, solved by Newton's method and compared with scikit-learn.
2. **Linear regression** (the base of our ridge loan-size model): least squares and the normal equation, plus the ridge closed form.
3. **Gradient boosting with log-loss** (the LightGBM challenger): why each tree fits $y - p$, and where the leaf values come from.

Notation: $n$ loans, design matrix $X \in \mathbb{R}^{n \times (d+1)}$ whose first column is ones, weights $\beta \in \mathbb{R}^{d+1}$, labels $y_i \in \{0, 1\}$ (1 = default).

In [1]:
import sys, warnings; sys.path.append(".."); warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from src.config import set_seed, DATA_PROCESSED
set_seed()
feat = pd.read_pickle(DATA_PROCESSED / "features.pkl")
tr = feat[feat.split == "train"].reset_index(drop=True)
# A small, fully observed numeric design keeps the checks exact (no one-hot collinearity, no imputation)
cols = ["duration", "n_prev_loans", "n_active_loans", "prev_default_rate", "Lender_portion_Funded", "n_lenders", "is_new", "disb_weekday"]
Z = StandardScaler().fit_transform(np.column_stack([np.log1p(tr.duration), np.log1p(tr.n_prev_loans)] + [tr[c] for c in cols[2:]]))
X = np.column_stack([np.ones(len(Z)), Z])
y = tr.target.to_numpy().astype(float)
X.shape

(66520, 9)

## 8.1 Logistic regression

**Model.** The probability of default is the sigmoid of a linear score:
$$p_i = \sigma(z_i) = \frac{1}{1 + e^{-z_i}}, \qquad z_i = x_i^\top \beta .$$
A useful identity: $\sigma'(z) = \sigma(z)\,(1 - \sigma(z))$, because
$\frac{d}{dz}(1 + e^{-z})^{-1} = \frac{e^{-z}}{(1 + e^{-z})^2} = \sigma(z)\cdot\frac{e^{-z}}{1 + e^{-z}} = \sigma(z)(1 - \sigma(z))$.

**Likelihood to log-loss.** Each label is Bernoulli: $P(y_i \mid x_i) = p_i^{y_i}(1 - p_i)^{1 - y_i}$. Assuming independent loans, maximising the likelihood equals minimising the average negative log-likelihood, the **log-loss**:
$$L(\beta) = -\frac{1}{n}\sum_{i=1}^{n}\Big[\, y_i \log p_i + (1 - y_i)\log(1 - p_i) \Big].$$

**Gradient.** By the chain rule, for one loan:
$$\frac{\partial \ell_i}{\partial p_i} = -\frac{y_i}{p_i} + \frac{1 - y_i}{1 - p_i} = \frac{p_i - y_i}{p_i(1 - p_i)}, \qquad
\frac{\partial p_i}{\partial z_i} = p_i(1 - p_i), \qquad \frac{\partial z_i}{\partial \beta} = x_i .$$
The $p_i(1-p_i)$ terms cancel, leaving
$$\nabla_\beta L = \frac{1}{n}\sum_{i=1}^{n}(p_i - y_i)\,x_i = \frac{1}{n}X^\top(p - y).$$
The gradient is the prediction error weighted by the features, the same form as in linear regression. Setting it to zero has no closed-form solution because $p$ depends non-linearly on $\beta$.

**Hessian and convexity.** Differentiating again, $\partial p_i / \partial \beta = p_i(1 - p_i)x_i$, so
$$\nabla^2_\beta L = \frac{1}{n}\sum_i p_i(1 - p_i)\,x_i x_i^\top = \frac{1}{n}X^\top W X, \qquad W = \operatorname{diag}\big(p_i(1 - p_i)\big).$$
For any $v$, $v^\top X^\top W X v = \sum_i p_i(1-p_i)(x_i^\top v)^2 \ge 0$, so the Hessian is positive semi-definite and **the log-loss is convex**: any minimum is the global minimum.

**Solving.** Newton's method (iteratively reweighted least squares) uses both:
$$\beta^{(t+1)} = \beta^{(t)} - \big(X^\top W X\big)^{-1} X^\top (p - y).$$
With L2 regularisation $\frac{\lambda}{2n}\lVert\beta_{1:d}\rVert^2$ (what scikit-learn's `C = 1/\lambda` controls), the gradient gains $\frac{\lambda}{n}\beta_{1:d}$ and the Hessian gains $\frac{\lambda}{n}I$ on the non-intercept terms.

**Class weights**, used in Section 5, multiply each loan's term by $w_{y_i}$, giving gradient $\frac{1}{n}X^\top \operatorname{diag}(w)(p - y)$. The optimum then satisfies a reweighted balance, so the fitted probabilities no longer estimate the true default rate. That is the algebraic reason the balanced model predicted an average P(default) of 0.29.

In [2]:
sigmoid = lambda z: 1 / (1 + np.exp(-z))
logloss = lambda b: -np.mean(y * np.log(sigmoid(X @ b)) + (1 - y) * np.log(1 - sigmoid(X @ b)))
grad = lambda b: X.T @ (sigmoid(X @ b) - y) / len(y)

# 1) Analytic gradient vs central finite differences at a random point
b0 = np.random.default_rng(0).normal(scale=0.1, size=X.shape[1])
eps = 1e-6
fd = np.array([(logloss(b0 + eps * e) - logloss(b0 - eps * e)) / (2 * eps) for e in np.eye(len(b0))])
print("max |analytic - finite difference| gradient:", f"{np.abs(grad(b0) - fd).max():.2e}")

# 2) Newton / IRLS from zero
b = np.zeros(X.shape[1])
for it in range(25):
    p = sigmoid(X @ b)
    H = X.T @ (X * (p * (1 - p))[:, None]) / len(y)
    step = np.linalg.solve(H, grad(b))
    b -= step
    if np.abs(step).max() < 1e-10:
        break
print(f"Newton converged in {it + 1} iterations; log-loss {logloss(b):.6f}; |gradient| {np.abs(grad(b)).max():.2e}")

# 3) Same model in scikit-learn (no penalty)
sk = LogisticRegression(penalty=None, max_iter=5000, tol=1e-12).fit(Z, y)
pd.DataFrame({"Newton (ours)": b, "scikit-learn": np.r_[sk.intercept_, sk.coef_.ravel()]},
             index=["intercept", "log duration", "log n_prev_loans"] + cols[2:]).round(5)

max |analytic - finite difference| gradient: 4.75e-11
Newton converged in 10 iterations; log-loss 0.067578; |gradient| 2.96e-17


,Newton (ours),scikit-learn
intercept,-4.91283,-4.91283
log duration,0.31766,0.31766
log n_prev_loans,-0.50102,-0.50102
n_active_loans,-0.93657,-0.93657
prev_default_rate,-0.07579,-0.07579
Lender_portion_Funded,0.16808,0.16808
n_lenders,0.10395,0.10395
is_new,0.09886,0.09886
disb_weekday,-0.03137,-0.03137


The analytic gradient matches finite differences to numerical precision, Newton converges in a handful of iterations (as expected for a convex, smooth loss), and the coefficients match scikit-learn. The signs match Section 3: longer terms raise risk, a longer borrowing history lowers it.

## 8.2 Linear regression and the normal equation

**Model and loss.** $\hat y = X\beta$, with squared error
$$J(\beta) = \frac{1}{2n}\lVert y - X\beta\rVert^2 = \frac{1}{2n}\big(y^\top y - 2\beta^\top X^\top y + \beta^\top X^\top X \beta\big).$$
Minimising squared error is maximum likelihood if $y_i = x_i^\top\beta + \varepsilon_i$ with $\varepsilon_i \sim \mathcal{N}(0, \sigma^2)$ independent: the Gaussian log-likelihood is $-\frac{1}{2\sigma^2}\lVert y - X\beta\rVert^2$ plus constants. Our target is $\log(1 + \text{amount})$; the Section 6 residuals were close to symmetric on that scale, apart from a negative tail, which is why the log scale is used.

**Gradient.** Using $\nabla_\beta(\beta^\top A \beta) = 2A\beta$ for symmetric $A$ and $\nabla_\beta(\beta^\top c) = c$:
$$\nabla_\beta J = \frac{1}{n}\big(X^\top X\beta - X^\top y\big) = -\frac{1}{n}X^\top(y - X\beta).$$
Again the features times the residual.

**Normal equation.** Setting the gradient to zero:
$$X^\top X \hat\beta = X^\top y \quad\Longrightarrow\quad \hat\beta = (X^\top X)^{-1}X^\top y,$$
valid when $X$ has full column rank. The Hessian $\frac{1}{n}X^\top X$ is positive semi-definite, so this stationary point is the global minimum. Geometrically, $X^\top(y - X\hat\beta) = 0$ says the residual is **orthogonal** to every column of $X$: the fit is the projection of $y$ onto the column space. In particular the residuals sum to zero (orthogonal to the column of ones).

**Ridge.** Adding $\frac{\lambda}{2n}\lVert\beta_{1:d}\rVert^2$ gives $\hat\beta = (X^\top X + \lambda D)^{-1}X^\top y$, where $D$ is the identity with a 0 for the intercept. $X^\top X + \lambda D$ is positive definite for any $\lambda > 0$: if $v^\top(X^\top X + \lambda D)v = \lVert Xv\rVert^2 + \lambda\lVert v_{1:d}\rVert^2 = 0$, then $v_{1:d} = 0$, and $Xv = v_0\mathbf{1} = 0$ forces $v_0 = 0$. That is why ridge stays stable with the correlated history features and one-hot columns used in Section 6. The check below shows that design is rank-deficient: one-hot columns for every level sum to the intercept column, so plain $X^\top X$ is singular and the normal equation has no unique solution.

**In practice** we solve $X^\top X\beta = X^\top y$ with a linear solver (or QR / least squares) rather than forming the inverse, which is slower and less accurate.

In [3]:
rp = tr.target.eq(0).to_numpy()
Xr, yr = X[rp], np.log1p(tr.Total_Amount.to_numpy()[rp])

beta_ne = np.linalg.solve(Xr.T @ Xr, Xr.T @ yr)              # normal equation
beta_ls = np.linalg.lstsq(Xr, yr, rcond=None)[0]              # QR/SVD least squares
lr = LinearRegression().fit(Xr[:, 1:], yr)
resid = yr - Xr @ beta_ne
print("max |normal eq - lstsq|:", f"{np.abs(beta_ne - beta_ls).max():.2e}",
      "| max |normal eq - sklearn|:", f"{np.abs(beta_ne - np.r_[lr.intercept_, lr.coef_]).max():.2e}")
print("orthogonality max |X^T r|:", f"{np.abs(Xr.T @ resid).max():.2e}", "| residual mean:", f"{resid.mean():.2e}")

lam = 10.0
D = np.eye(Xr.shape[1]); D[0, 0] = 0
beta_ridge = np.linalg.solve(Xr.T @ Xr + lam * D, Xr.T @ yr)
rg = Ridge(alpha=lam).fit(Xr[:, 1:], yr)
print("max |ridge closed form - sklearn Ridge|:", f"{np.abs(beta_ridge - np.r_[rg.intercept_, rg.coef_]).max():.2e}")
print("small design: condition number of X^T X", f"{np.linalg.cond(Xr.T @ Xr):.1f}", "(already well conditioned, ridge changes little)")

# The real Section 6 design: one-hot columns for every level plus an intercept are collinear
from src import models as M
prep = M.preprocessor(M.REG_FEATURES, scale=True, log_cols=M.LOG_TERMS)
Xf = prep.fit_transform(tr.loc[rp, M.inputs(M.REG_FEATURES)])
Xf = np.column_stack([np.ones(len(Xf)), Xf])
G = Xf.T @ Xf
Df = np.eye(G.shape[0]); Df[0, 0] = 0
print(f"Section 6 design: {Xf.shape[1]} columns, rank {np.linalg.matrix_rank(Xf)}; cond(X^T X) = {np.linalg.cond(G):.1e}"
      f" -> with ridge (lambda = 10): {np.linalg.cond(G + 10 * Df):.1e}")

max |normal eq - lstsq|: 1.08e-12 | max |normal eq - sklearn|: 1.08e-12
orthogonality max |X^T r|: 7.36e-08 | residual mean: 1.19e-14
max |ridge closed form - sklearn Ridge|: 1.11e-12
small design: condition number of X^T X 6.9 (already well conditioned, ridge changes little)


Section 6 design: 40 columns, rank 35; cond(X^T X) = 1.7e+19 -> with ridge (lambda = 10): 5.8e+04


## 8.3 Gradient boosting with log-loss (the LightGBM challenger)

Boosting builds an additive score $F_M(x) = F_0 + \sum_{m=1}^{M} \eta\, f_m(x)$ of small trees $f_m$, with $p = \sigma(F)$, by **gradient descent in function space**.

**Start.** The best constant minimises the log-loss: setting $\frac{\partial}{\partial F_0}\sum_i \ell(y_i, F_0) = \sum_i(\sigma(F_0) - y_i) = 0$ gives $\sigma(F_0) = \bar y$, so
$$F_0 = \log\frac{\bar y}{1 - \bar y},$$
the log-odds of the base default rate. That is exactly the baseline model of Section 5.

**Each round fits the negative gradient.** From 8.1, the derivative of one loan's loss with respect to its *score* is $\partial \ell_i / \partial F_i = p_i - y_i$. The steepest-descent direction for each loan is therefore the **pseudo-residual**
$$r_i = y_i - p_i,$$
and tree $m$ is grown to predict $r_i$ from $x_i$. Defaulters the model currently under-rates ($y = 1$, low $p$) get large positive targets.

**Leaf values (Newton step).** The second derivative is $h_i = \partial^2\ell_i/\partial F_i^2 = p_i(1 - p_i)$, the same weight as in the logistic Hessian. For a leaf $R$ containing loans $i \in R$, a second-order Taylor expansion of the loss in the leaf value $\gamma$,
$$\sum_{i\in R}\ell(y_i, F_i + \gamma) \approx \sum_{i\in R}\Big[\ell(y_i, F_i) + g_i\gamma + \tfrac12 h_i\gamma^2\Big], \qquad g_i = p_i - y_i,$$
is minimised at
$$\gamma_R = -\frac{\sum_{i\in R} g_i}{\sum_{i\in R} h_i + \lambda} = \frac{\sum_{i\in R}(y_i - p_i)}{\sum_{i\in R}p_i(1 - p_i) + \lambda},$$
where $\lambda$ is LightGBM's `reg_lambda`. Substituting back, the loss reduction from a leaf is $\frac{1}{2}\frac{(\sum g_i)^2}{\sum h_i + \lambda}$, and LightGBM chooses splits that maximise the gain $\frac12\big[\frac{G_L^2}{H_L+\lambda} + \frac{G_R^2}{H_R+\lambda} - \frac{(G_L+G_R)^2}{H_L+H_R+\lambda}\big]$.

**Link to our results.** `min_child_samples` limits how few loans a leaf may use (our shallow model required 200), and `num_leaves` limits tree size. With only a few hundred defaults, small leaves can fit noise in the 2022 boom, which is what Sections 4 and 5 saw out of time. Logistic regression is the special case of a model that is linear in $x$, optimised with the same gradient $X^\top(p - y)$.

In [4]:
# Verify: one boosting round by hand, a depth-1 stump on log duration, against the formulas
ybar = y.mean(); F0 = np.log(ybar / (1 - ybar)); p0 = np.full_like(y, ybar)
g, h = p0 - y, p0 * (1 - p0)
x = Z[:, 0]                                   # standardised log duration
cands = np.quantile(x, np.linspace(0.05, 0.95, 37))
lam = 1.0
def gain(t):
    L = x <= t; GL, HL, GR, HR = g[L].sum(), h[L].sum(), g[~L].sum(), h[~L].sum()
    return 0.5 * (GL**2 / (HL + lam) + GR**2 / (HR + lam) - (GL + GR)**2 / (HL + HR + lam))
t_best = cands[np.argmax([gain(t) for t in cands])]
L = x <= t_best
gamma = {side: -g[m].sum() / (h[m].sum() + lam) for side, m in [("short", L), ("long", ~L)]}
F1 = F0 + np.where(L, gamma["short"], gamma["long"])
loss = lambda F: -np.mean(y * np.log(sigmoid(F)) + (1 - y) * np.log(1 - sigmoid(F)))
print(f"F0 = log-odds of base rate = {F0:.4f}  (sigmoid -> {sigmoid(F0):.4f}, base rate {ybar:.4f})")
print(f"Best split on standardised log duration at {t_best:.3f}; leaf values {gamma['short']:.3f} (short) and {gamma['long']:.3f} (long)")
print(f"Log-loss {loss(np.full_like(y, F0)):.5f} -> {loss(F1):.5f} after one Newton-boosted stump")
print("Default rate by leaf:", {k: round(y[m].mean(), 4) for k, m in [("short", L), ("long", ~L)]},
      "| model p by leaf:", {k: round(float(sigmoid(F0 + gamma[k])), 4) for k in gamma})

F0 = log-odds of base rate = -4.1023  (sigmoid -> 0.0163, base rate 0.0163)
Best split on standardised log duration at -0.179; leaf values -0.343 (short) and 7.727 (long)
Log-loss 0.08313 -> 0.18750 after one Newton-boosted stump
Default rate by leaf: {'short': np.float64(0.0108), 'long': np.float64(0.1427)} | model p by leaf: {'short': 0.0116, 'long': 0.974}


The split found is the term boundary, the strongest single signal in Section 3. But the **full Newton step overshoots** on the long-term leaf: it jumps to p ≈ 0.97 when the leaf's default rate is 0.14, and log-loss *rises*. The reason is in the formula: starting from the base rate, the curvature $h_i = p_i(1-p_i) pprox 0.016$ is tiny, so the quadratic approximation badly underestimates how fast the loss turns up as $p$ grows. With rare events, that is the normal case.

This is why boosting uses a **learning rate** $\eta$ (ours is 0.03) and many rounds, recomputing $g$ and $h$ each time. The cell below repeats the same stump: with full steps ($\eta = 1$) the updates oscillate and diverge (the loss becomes undefined as probabilities hit 0 or 1); with shrinkage each round moves partway, the curvature updates, and both leaves converge to their observed default rates while log-loss falls monotonically. (LightGBM also offers `max_delta_step` to cap a leaf's step for exactly this situation.)

In [5]:
def boost(eta, rounds):
    F = np.full_like(y, F0); hist = [loss(F)]
    for _ in range(rounds):
        p = sigmoid(F); g, h = p - y, p * (1 - p)
        for m in (L, ~L):
            F[m] += eta * (-g[m].sum() / (h[m].sum() + lam))
        hist.append(loss(F))
    return F, hist
rows = []
for eta, rounds in [(1.0, 1), (1.0, 20), (0.3, 20), (0.1, 60)]:
    F, hist = boost(eta, rounds)
    rows.append({"learning rate": eta, "rounds": rounds, "final log-loss": hist[-1], "loss ever rose": bool(np.any(np.diff(hist) > 1e-12)),
                 "p short leaf": sigmoid(F[L][0]), "p long leaf": sigmoid(F[~L][0])})
out = pd.DataFrame(rows)
out.loc[len(out)] = {"learning rate": "observed", "rounds": "", "final log-loss": loss(np.where(L, np.log(y[L].mean() / (1 - y[L].mean())), np.log(y[~L].mean() / (1 - y[~L].mean())))),
                     "loss ever rose": "", "p short leaf": y[L].mean(), "p long leaf": y[~L].mean()}
out.round(4)

,learning rate,rounds,final log-loss,loss ever rose,p short leaf,p long leaf
0,1.0,1,0.1875,True,0.0116,0.9740
1,1.0,20,NaN,True,0.0108,0.0000
2,0.3,20,0.0741,False,0.0108,0.1427
3,0.1,60,0.0741,False,0.0108,0.1425
4,observed,,0.0741,,0.0108,0.1427
